# Orchestrator Agent with Sub-Agents

This demo creates two specialist Microsoft Foundry agents and an orchestrator agent. The orchestrator receives the specialists' outputs and produces one implementation-ready answer.

In [1]:
%pip install -q azure-ai-projects==2.0.0b2 azure-identity python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load the existing project configuration

The notebook reuses the current details from `A2A/A2A_and_MCP/.env`.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity.aio import DefaultAzureCredential

env_path = (Path.cwd().parent / "A2A_and_MCP" / ".env").resolve()
load_dotenv(env_path)

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

assert foundry_project_endpoint, f"FOUNDRY_PROJECT_ENDPOINT was not found in {env_path}"
assert model_deployment_name, f"MODEL_DEPLOYMENT_NAME was not found in {env_path}"

print(f"Loaded configuration from: {env_path}")
print(f"Model deployment: {model_deployment_name}")

Loaded configuration from: D:\TRAININGS_Recent_Sessions\EY_Agentic_AI_Level3-May-June2026\udmy\MicrosoftAI-Foundry-main\A2A\A2A_and_MCP\.env
Model deployment: ajay-gpt-4o


## Connect to Foundry and create the agent team

In [3]:
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

async def create_prompt_agent(name: str, instructions: str):
    agent = await project_client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=instructions,
        ),
    )
    print(f"Created {agent.name} version {agent.version}")
    return agent

async def invoke_agent(agent, prompt: str) -> str:
    conversation = await openai_client.conversations.create()
    response = await openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        input=prompt,
    )
    return response.output_text

requirements_agent = await create_prompt_agent(
    "orchestration-requirements-agent",
    "You are a requirements analyst. Identify goals, assumptions, constraints, risks, and acceptance criteria. Be concise and concrete.",
)

implementation_agent = await create_prompt_agent(
    "orchestration-implementation-agent",
    "You are a senior Azure implementation specialist. Produce practical implementation steps, commands, security guidance, and verification checks.",
)

orchestrator_agent = await create_prompt_agent(
    "orchestration-manager-agent",
    "You are the orchestrator. Synthesize specialist reports into one accurate, non-redundant, implementation-ready answer. Call out unresolved assumptions.",
)

Created orchestration-requirements-agent version 1
Created orchestration-implementation-agent version 1
Created orchestration-manager-agent version 1


## Run the orchestration

The two sub-agents work independently. Their reports are then delegated to the orchestrator for synthesis.

In [4]:
user_request = "Design a secure Azure Storage Account deployment using Azure CLI for a production application."

requirements_report = await invoke_agent(requirements_agent, user_request)
implementation_report = await invoke_agent(implementation_agent, user_request)

orchestrator_prompt = f"""Create the final response for this request:
{user_request}

Requirements analyst report:
{requirements_report}

Implementation specialist report:
{implementation_report}

Return a concise architecture summary, ordered implementation steps, Azure CLI commands, security controls, and validation checklist.
"""

final_answer = await invoke_agent(orchestrator_agent, orchestrator_prompt)
display(Markdown("## Requirements Sub-Agent\n" + requirements_report))
display(Markdown("## Implementation Sub-Agent\n" + implementation_report))
display(Markdown("## Orchestrator Final Answer\n" + final_answer))

## Requirements Sub-Agent
**Goals**:  
1. Deploy a secure Azure Storage Account optimized for production use.  
2. Implement security best practices for data at rest and in transit.  
3. Utilize Azure CLI for efficient and repeatable deployment.  
4. Ensure compliance with organizational and industry security standards.  

**Assumptions**:  
1. The organization has access to the required Azure subscription and permissions.  
2. Azure CLI is installed and configured on the local system.  
3. The application requires high availability and durability for data storage.  
4. Security features like encryption, private endpoints, and access policies are prioritized.  

**Constraints**:  
1. Deployment must adhere to Azure's limits for storage accounts (e.g., request rates, account quotas).  
2. Network traffic between the storage account and the application must use secure channels (e.g., HTTPS).  
3. Budget limitations require cost optimization while maintaining security and performance.  

**Risks**:  
1. Misconfiguration of the storage account could expose sensitive data.  
2. Network security risks (e.g., allowing public traffic) could impact data confidentiality.  
3. Outdated CLI version or invalid subscription permissions could prevent deployment.  
4. Downtime or service disruptions during missteps in production implementation.  

**Acceptance Criteria**:  
1. Azure Storage Account deployed with encryption enabled (e.g., SSE, customer-managed keys).  
2. All traffic to and from the storage account occurs over HTTPS.  
3. Default access to the storage account is restricted through firewall rules and/or private endpoint configuration.  
4. Storage account operations are verified as successful via Azure CLI commands and integration with the application.  
5. Deployment documentation is provided, outlining steps to recreate or modify the storage account setup.  

## Implementation Sub-Agent
Below is a comprehensive step-by-step guide to designing and implementing a secure Azure Storage Account deployment using Azure CLI for a production application. This includes practical commands, security best practices, and verification checks.

---

### **Prerequisites**
1. **Azure CLI Installed**: Ensure Azure CLI (`az`) is installed and authenticated (`az login`).
2. **Subscription Set**: Set the correct subscription:
   ```bash
   az account set --subscription <SUBSCRIPTION_ID>
   ```

---

### **1. Create a Resource Group**
A resource group organizes resources logically.

```bash
az group create --name <RESOURCE_GROUP_NAME> --location <LOCATION>
```

- Replace `<RESOURCE_GROUP_NAME>` with a descriptive name (e.g., `prod-storage-rg`).
- Replace `<LOCATION>` with your Azure region (e.g., `eastus`).

---

### **2. Create the Storage Account**
Securely create a storage account with production-worthy configurations.

```bash
az storage account create \
    --name <STORAGE_ACCOUNT_NAME> \
    --resource-group <RESOURCE_GROUP_NAME> \
    --location <LOCATION> \
    --sku Standard_LRS \
    --encryption-services blob \
    --access-tier Hot \
    --kind StorageV2 \
    --enable-hierarchical-namespace true \
    --allow-blob-public-access false \
    --min-tls-version TLS1_2 \
    --https-only true
```

#### Breakdown:
- `--sku Standard_LRS`: Local redundant storage (change to `Standard_GRS` for geo-redundancy if needed).
- `--encryption-services blob`: Enable encryption for blobs.
- `--access-tier Hot`: Optimize for frequent access; change to `Cool` for infrequent access.
- `--kind StorageV2`: Enables advanced features like hierarchical namespace and blob storage.
- `--enable-hierarchical-namespace true`: Required for Azure Data Lake functionality.
- `--allow-blob-public-access false`: Disable public access for blobs (secure default).
- `--min-tls-version TLS1_2`: Enforce secure TLS versions.
- `--https-only true`: Force HTTPS traffic.

---

### **3. Configure Networking Restriction**
Restrict access to the storage account from specific IPs or Virtual Networks.

#### Enable Firewall and Virtual Network:
```bash
az storage account update \
    --name <STORAGE_ACCOUNT_NAME> \
    --resource-group <RESOURCE_GROUP_NAME> \
    --default-action Deny
```

#### Add a Trusted IP Rule:
```bash
az storage account network-rule add \
    --resource-group <RESOURCE_GROUP_NAME> \
    --account-name <STORAGE_ACCOUNT_NAME> \
    --action Allow \
    --ip-address <TRUSTED_IP>
```

- Replace `<TRUSTED_IP>` with your trusted IP, e.g., `203.0.113.0/32`.

#### Add a Virtual Network Rule:
```bash
az storage account network-rule add \
    --resource-group <RESOURCE_GROUP_NAME> \
    --account-name <STORAGE_ACCOUNT_NAME> \
    --action Allow \
    --vnet-name <VNET_NAME> \
    --subnet <SUBNET_NAME>
```

Ensure the subnet is properly associated with the Virtual Network.

---

### **4. Set Secure Access Keys and Shared Access Token (SAS) Policies**
Access to the storage account is secured with Access Keys and SAS policies.

#### Rotate Access Keys Regularly:
```bash
az storage account keys renew \
    --resource-group <RESOURCE_GROUP_NAME> \
    --account-name <STORAGE_ACCOUNT_NAME> \
    --key primary
```

#### Generate a Time-Bound SAS Token for Secure Access:
```bash
az storage account generate-sas \
    --permissions rwl \
    --expiry <YYYY-MM-DDTHH:MM:SSZ> \
    --resource-types sco \
    --services b \
    --account-name <STORAGE_ACCOUNT_NAME> \
    --https-only
```

---

### **5. Enable Diagnostic Logs**
Enable resource-level logging to monitor storage activities.

#### Enable Diagnostic Settings:
```bash
az monitor diagnostic-settings create \
    --name <DIAGNOSTIC_NAME> \
    --resource-group <RESOURCE_GROUP_NAME> \
    --resource <STORAGE_ACCOUNT_NAME> \
    --resource-type "Microsoft.Storage/storageAccounts" \
    --workspace <LOG_ANALYTICS_WORKSPACE_ID> \
    --logs '[{"category": "StorageWrite", "enabled": true}]' \
    --metrics '[{"category": "Transaction", "enabled": true}]'
```

- Replace `<LOG_ANALYTICS_WORKSPACE_ID>` with your Log Analytics workspace ID.

---

### **6. Enable Azure Private Endpoint**
Secure access to the storage account over private network.

#### Create a Private Endpoint:
```bash
az network private-endpoint create \
    --name <PRIVATE_ENDPOINT_NAME> \
    --resource-group <RESOURCE_GROUP_NAME> \
    --vnet-name <VNET_NAME> \
    --subnet <SUBNET_NAME> \
    --connection-name <PE_CONNECTION_NAME> \
    --private-connection-resource-id $(az storage account show --name <STORAGE_ACCOUNT_NAME> --resource-group <RESOURCE_GROUP_NAME> --query id --output tsv) \
    --group-id blob
```

#### Approve Private Endpoint Connection:
You may need to manually approve the private endpoint connection in the Azure portal depending on your setup.

---

### **7. Verify Deployment**
Execute verification checks:

#### Check Storage Account Configuration:
```bash
az storage account show --name <STORAGE_ACCOUNT_NAME> --resource-group <RESOURCE_GROUP_NAME>
```

#### Check Network Rules:
```bash
az storage account network-rule list --name <STORAGE_ACCOUNT_NAME> --resource-group <RESOURCE_GROUP_NAME>
```

#### Verify Diagnostic Logs:
```bash
az monitor diagnostic-settings list \
    --resource-group <RESOURCE_GROUP_NAME> \
    --resource <STORAGE_ACCOUNT_NAME>
```

---

### **8. Monitor and Secure Account**
1. **Enable Azure Policy**: Apply policies to enforce security standards, like restricting public access.
2. **Enable Azure Defender for Storage**:
   ```bash
   az security contact create \
       --email <YOUR_EMAIL> \
       --phone <YOUR_PHONE> \
       --alert-notifications enabled \
       --alerts-to-manager enabled
   ```

3. **Regular Audit**: Periodically rotate access keys, review SAS tokens, and analyze diagnostic logs for suspicious activity.

---

### **Final Notes**
1. Follow the principle of least privilege—restrict access to only those users, applications, networks, and IPs that need it.
2. Regularly review security settings and monitor for unusual activities.
3. Use automation tools like Azure DevOps or Terraform for repeatable deployments.



## Orchestrator Final Answer
### **Secure Azure Storage Account Deployment Architecture Summary**
A secure Azure Storage Account is configured using Azure CLI for production applications with the following characteristics:
- **Type**: StorageV2 for advanced features (hierarchical namespace, encryption).
- **Redundancy**: Standard_LRS by default; configurable to Standard_GRS (geo-redundancy).
- **Encryption**: Enabled with server-side encryption (SSE), supporting Azure-managed or customer-managed keys.
- **Network Isolation**: Firewall rules, private endpoints, and HTTPS-only access ensure secure communication.
- **Access Control**: Fine-grained access via Shared Access Signatures (SAS), regular key rotation.
- **Monitoring**: Diagnostic logs integrated with Log Analytics for auditing and anomaly detection.

---

### **Ordered Implementation Steps**
**1. Prerequisites**:  
   - Ensure Azure CLI is installed, authenticated, and configured (`az login` and `az account set`).

**2. Resource Group Deployment**:  
   ```bash
   az group create --name <RESOURCE_GROUP_NAME> --location <LOCATION>
   ```

**3. Storage Account Creation**:  
   ```bash
   az storage account create \
       --name <STORAGE_ACCOUNT_NAME> \
       --resource-group <RESOURCE_GROUP_NAME> \
       --location <LOCATION> \
       --sku Standard_LRS \
       --encryption-services blob \
       --access-tier Hot \
       --kind StorageV2 \
       --enable-hierarchical-namespace true \
       --allow-blob-public-access false \
       --min-tls-version TLS1_2 \
       --https-only true
   ```

**4. Networking Restrictions**:  
   - **Default Deny Policy**:  
     ```bash
     az storage account update \
         --name <STORAGE_ACCOUNT_NAME> \
         --resource-group <RESOURCE_GROUP_NAME> \
         --default-action Deny
     ```
   - **Add Trusted IPs**:
     ```bash
     az storage account network-rule add \
         --resource-group <RESOURCE_GROUP_NAME> \
         --account-name <STORAGE_ACCOUNT_NAME> \
         --action Allow \
         --ip-address <TRUSTED_IP>
     ```
   - **Add Virtual Network Rule**:  
     ```bash
     az storage account network-rule add \
         --resource-group <RESOURCE_GROUP_NAME> \
         --account-name <STORAGE_ACCOUNT_NAME> \
         --action Allow \
         --vnet-name <VNET_NAME> \
         --subnet <SUBNET_NAME>
     ```

**5. Access Key Management**:  
   - Rotate Access Keys Regularly:  
     ```bash
     az storage account keys renew \
         --resource-group <RESOURCE_GROUP_NAME> \
         --account-name <STORAGE_ACCOUNT_NAME> \
         --key primary
     ```
   - Generate SAS Tokens as needed:  
     ```bash
     az storage account generate-sas \
         --permissions rwl \
         --expiry <YYYY-MM-DDTHH:MM:SSZ> \
         --resource-types sco \
         --services b \
         --account-name <STORAGE_ACCOUNT_NAME> \
         --https-only
     ```

**6. Diagnostic Logging**:  
   - Enable monitoring for auditing:  
     ```bash
     az monitor diagnostic-settings create \
         --name <DIAGNOSTIC_NAME> \
         --resource-group <RESOURCE_GROUP_NAME> \
         --resource <STORAGE_ACCOUNT_NAME> \
         --resource-type "Microsoft.Storage/storageAccounts" \
         --workspace <LOG_ANALYTICS_WORKSPACE_ID> \
         --logs '[{"category": "StorageWrite", "enabled": true}]' \
         --metrics '[{"category": "Transaction", "enabled": true}]'
     ```

**7. Private Endpoint Configuration**:  
   - Create endpoint for private network access:  
     ```bash
     az network private-endpoint create \
         --name <PRIVATE_ENDPOINT_NAME> \
         --resource-group <RESOURCE_GROUP_NAME> \
         --vnet-name <VNET_NAME> \
         --subnet <SUBNET_NAME> \
         --connection-name <PE_CONNECTION_NAME> \
         --private-connection-resource-id $(az storage account show --name <STORAGE_ACCOUNT_NAME> --resource-group <RESOURCE_GROUP_NAME> --query id --output tsv) \
         --group-id blob
     ```
   - Approve manually if required.

**8. Validation Checks**:  
   - Verify Storage Account:  
     ```bash
     az storage account show --name <STORAGE_ACCOUNT_NAME> --resource-group <RESOURCE_GROUP_NAME>
     ```
   - Check Network Rules:  
     ```bash
     az storage account network-rule list --name <STORAGE_ACCOUNT_NAME> --resource-group <RESOURCE_GROUP_NAME>
     ```
   - Verify Diagnostic Settings:  
     ```bash
     az monitor diagnostic-settings list \
         --resource-group <RESOURCE_GROUP_NAME> \
         --resource <STORAGE_ACCOUNT_NAME>
     ```

---

### **Security Controls**
1. Enforced encryption at rest (SSE with Azure-managed keys).
2. TLS 1.2 minimum for secure in-transit communication.
3. HTTPS-only access.
4. Disabled blob public access by default.
5. Firewall rules and IP/VNet restrictions.
6. SAS tokens for time-limited access.
7. Integrated private endpoint for internal traffic security.

---

### **Validation Checklist**
1. **Security**:  
   - Encryption enabled (`encryption-services blob`).  
   - TLS minimum version is 1.2 (`min-tls-version TLS1_2`).  
   - HTTPS-only traffic (`https-only true`).

2. **Networking**:  
   - Firewall rules default deny access.  
   - Virtual Network or IP rules applied correctly.  
   - Private Endpoint showing approved connection status.

3. **Access Control**:  
   - Access keys rotated per policy.  
   - SAS tokens generated securely with time-bound permissions.

4. **Monitoring**:  
   - Diagnostic logs flowing into Log Analytics.  
   - Storage metrics enabled for auditing and alerts.

5. **Operational Tests**:  
   - Application integrated and able to perform storage operations (read/write).  
   - CLI commands return correct configuration data.

---

### **Unresolved Assumptions**
1. The configuration of VNet/Subnet for private endpoints will vary based on the organization’s existing network topology.
2. Industry-specific compliance standards (e.g., HIPAA, GDPR) may require additional configurations like customer-managed keys and Azure Policy assignments.  
3. There may be budgetary implications for enabling geo-redundancy or private endpoints that are not explicitly addressed.  


## Close clients

In [5]:
await project_client.close()
await credential.close()
print("Clients closed.")

Clients closed.
